# 02 - Analyser les resultats et les scores

Ce notebook relit `outputs/analysis/results.json` et aide a prioriser les controles humains, notamment les alertes generiques configurees et les alertes Article 9 RGPD.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
elif not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from compliance_nlp import load_results, results_to_dataframe, summarize_results

import pandas as pd

## Chargement des resultats

In [ ]:
results_path = repo_root / "outputs" / "analysis" / "results.json"
exports_dir = repo_root / "outputs" / "analysis"

results = load_results(results_path)
df = results_to_dataframe(results)
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["is_article9"] = df["rule_scope"].fillna("").eq("article9")
df["is_generic"] = df["rule_id"].notna() & df["rule_scope"].fillna("").ne("article9")

summary = summarize_results(results)
summary

## Vue globale des alertes

In [ ]:
overview_cols = [
    "document_name",
    "code",
    "severity",
    "alert_level",
    "section",
    "rule_scope",
    "regulatory_family",
    "category",
    "matched_term",
    "detection_type",
    "score",
    "rule_id",
]
df.sort_values(
    ["is_article9", "is_generic", "score", "severity"],
    ascending=[False, False, False, True],
    na_position="last",
)[overview_cols]

## Tableaux de synthese

In [ ]:
df.groupby(["code", "severity"], dropna=False).size().reset_index(name="count").sort_values("count", ascending=False)

In [ ]:
df.groupby(["document_name"], dropna=False).agg(
    finding_count=("code", lambda values: values.notna().sum()),
    max_score=("score", "max"),
    article9_count=("is_article9", "sum"),
).reset_index().sort_values(["article9_count", "max_score", "finding_count"], ascending=False)

## Focus controles generiques

In [ ]:
generic_df = df[df["is_generic"]].copy()
generic_df[[
    "document_name",
    "section",
    "rule_scope",
    "regulatory_family",
    "category",
    "rule_id",
    "matched_term",
    "detection_type",
    "score",
    "severity",
    "evidence",
]].sort_values(["score", "category"], ascending=[False, True], na_position="last")

In [ ]:
generic_df.groupby(["section", "category", "detection_type"], dropna=False).agg(
    count=("code", "size"),
    max_score=("score", "max"),
    avg_score=("score", "mean"),
).reset_index().sort_values(["max_score", "count"], ascending=False)

## Focus Article 9 RGPD

In [ ]:
article9_df = df[df["is_article9"]].copy()
article9_df[[
    "document_name",
    "category",
    "regulatory_family",
    "rule_id",
    "matched_term",
    "detection_type",
    "score",
    "severity",
    "evidence",
]].sort_values(["score", "category"], ascending=[False, True], na_position="last")

In [ ]:
article9_df.groupby(["category", "detection_type"], dropna=False).agg(
    count=("code", "size"),
    max_score=("score", "max"),
    avg_score=("score", "mean"),
).reset_index().sort_values(["max_score", "count"], ascending=False)

## Seuils de revue humaine

In [ ]:
HUMAN_REVIEW_SCORE = 0.65
CORRECTION_SCORE = 0.85

def review_action(score):
    if pd.isna(score):
        return "hors scoring"
    if score >= CORRECTION_SCORE:
        return "controle humain / correction"
    if score >= HUMAN_REVIEW_SCORE:
        return "controle humain"
    return "journalisation"

df["review_action"] = df["score"].apply(review_action)
df.groupby("review_action", dropna=False).size().reset_index(name="count")

In [ ]:
df[df["review_action"].isin(["controle humain", "controle humain / correction"])][[
    "document_name",
    "category",
    "rule_id",
    "matched_term",
    "detection_type",
    "score",
    "review_action",
    "evidence",
]].sort_values("score", ascending=False)

## Analyse des faux positifs potentiels

In [ ]:
df[df["detection_type"].isin(["fuzzy", "root", "synonym"])][[
    "document_name",
    "category",
    "rule_id",
    "matched_term",
    "detection_type",
    "score",
    "evidence",
]].sort_values(["detection_type", "score"], ascending=[True, False])

## Exports pour revue

In [ ]:
exports_dir.mkdir(parents=True, exist_ok=True)

all_findings_csv = exports_dir / "findings_analysis.csv"
generic_csv = exports_dir / "generic_findings.csv"
article9_csv = exports_dir / "article9_findings.csv"
human_review_csv = exports_dir / "human_review_queue.csv"

df.to_csv(all_findings_csv, index=False, encoding="utf-8-sig")
generic_df.to_csv(generic_csv, index=False, encoding="utf-8-sig")
article9_df.to_csv(article9_csv, index=False, encoding="utf-8-sig")
df[df["review_action"].isin(["controle humain", "controle humain / correction"])].to_csv(
    human_review_csv,
    index=False,
    encoding="utf-8-sig",
)

{
    "all_findings_csv": str(all_findings_csv),
    "generic_csv": str(generic_csv),
    "article9_csv": str(article9_csv),
    "human_review_csv": str(human_review_csv),
}